In [ ]:
!pip install -U transformers datasets accelerate peft bitsandbytes trl
!pip install protobuf==3.20.3 "pyarrow<20.0.0"

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [ ]:
raw_datasets = load_dataset("ailsntua/QEvasion")
test_dataset = raw_datasets['test']

In [ ]:
model_name = "gabrielstefan04/llama-3-1-8b-clarity"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

In [ ]:
def evaluate_model_batched(model, tokenizer, dataset, batch_size):
    predictions = []
    true_labels = []
    
    prompts = []
    for example in tqdm(dataset):
        conversation = [
            {
                "role": "system",
                "content": "You are an expert in analyzing political discourse. Classify the clarity of responses into: Clear Reply, Ambivalent, or Clear Non-Reply. Output ONLY the label name. Do not explain."
            },
            {
                "role": "user",
                "content": f"Question: {example['question']}\nAnswer: {example['interview_answer']}\n\nClassify the clarity of this response."
            }
        ]
        
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=True
        )
        prompts.append(text)
        true_labels.append(example['clarity_label'])
    
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i : i + batch_size]
        
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
        
        generated_texts = tokenizer.batch_decode(
            outputs[:, inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        
        for text in generated_texts:
            clean_text = text.strip()
            
            if "Clear Reply" in clean_text or "clear reply" in clean_text.lower():
                pred = "Clear Reply"
            elif "Ambivalent" in clean_text or "ambivalent" in clean_text.lower():
                pred = "Ambivalent"
            elif "Clear Non-Reply" in clean_text or "clear non-reply" in clean_text.lower():
                pred = "Clear Non-Reply"
            else:
                pred = "Ambivalent"  
            
            predictions.append(pred)
    
    return predictions, true_labels

In [ ]:
predictions, true_labels = evaluate_model_batched(model, tokenizer, test_dataset, batch_size=2)

In [ ]:
accuracy = accuracy_score(true_labels, predictions)
macro_f1 = f1_score(
    true_labels,
    predictions,
    average='macro',
    labels=["Clear Reply", "Ambivalent", "Clear Non-Reply"],
    zero_division=0
)
print("Subtask 1 - LLama 3.1 8B Clarity Classification")
print(f"\nAccuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print("Classification Report:")
print(classification_report(
    true_labels,
    predictions,
    labels=["Clear Reply", "Ambivalent", "Clear Non-Reply"],
    zero_division=0
))
print("Confusion Matrix:")
cm = confusion_matrix(
    true_labels,
    predictions,
    labels=["Clear Reply", "Ambivalent", "Clear Non-Reply"]
)
print(pd.DataFrame(
    cm,
    index=["Clear Reply", "Ambivalent", "Clear Non-Reply"],
    columns=["Clear Reply", "Ambivalent", "Clear Non-Reply"]
))